# Experiment Notebook
This note will perform the following :
* Load TypiClust Selections
* Generate random selections
* Train ResNet-18
* Save Results 
* Generate Diagrams 
* Perform statistical analysis - entailing, Paired t-tests, Confidence Intervals and Effect Sizes


# Modified Variant [ Commit 42 - current ] 
* The modified variant will include features listed above, but will also feature a *three-way* comparison between TypiClust, TypiClust-HDBSCAN & Random 
* Please ensure training is done using 500 epochs on both Typiclust & its variant 

In [ ]:
import os 
import numpy as np
import torch 
import torchvision
import random 

# Graphical concerns below
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd 
from tqdm import tqdm  # Loading visualiser.

from supervised_training import train_supervised




In [ ]:
# Reproducibility measures
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# budgets matching the paper 
budgets = [10,20,40,80]
seeds= [0,1,2,3,4] # Statistical Inference

In [ ]:
typiclust_kmeans = {
    B : np.load(f"../TPCRP_Algorithm/budget_results/unmodified_budget_results/typiclust_B{B}.npy")
    for B in budgets
}

typiclust_hdbscan = {
    B : np.load(f"../TPCRP_Algorithm/budget_results/modified_budget_results/hdbscan_results/typiclust_HDBSCAN_B{B}.npy")
    for B in budgets 
}


In [ ]:
# Alternatively to perform inference on the unsupervised_kmeans 

typiclust_unsupervised = {
    B : np.load(f"../TPCRP_Algorithm/unsupervised_budget_results/typiclust_B{B}.npy")
    for B in budgets 
}

In [ ]:
random_selections = {}

# For the random sampling to be done in conjunc. with TypiClust-HDBSCAN & TypiClust Regular 
for seed in seeds:
    np.random.seed(seed)
    random_selections[seed] = {
        B: np.random.choice(50000, size=B, replace=False)
        for B in budgets
    }


In [ ]:
os.makedirs("unsupervised_results/accuracies", exist_ok=True)
os.makedirs("unsupervised_results/loss_curves", exist_ok=True)
os.makedirs("unsupervised_results/runtimes", exist_ok=True)

In [ ]:
os.makedirs("results/accuracies", exist_ok=True)
os.makedirs("results/loss_curves", exist_ok=True)
os.makedirs("results/runtimes", exist_ok=True)


In [ ]:
results = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    for B in budgets:

        # ---- Unsupervised TypiClust ----
        typi_idx = typiclust_unsupervised[B]

        acc, loss_curve, runtime = train_supervised(
            selected_indices=typi_idx,
            epochs=50,
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )

        results.append({
            "method": "TypiClust-Unsupervised",
            "budget": B,
            "seed": seed,
            "accuracy": acc,
            "runtime": runtime
        })

        np.save(f"unsupervised_results/loss_curves/typiclust_B{B}_seed{seed}.npy",
                np.array(loss_curve))


        # ---- Random baseline ----
        rand_idx = random_selections[seed][B]

        acc, loss_curve, runtime = train_supervised(
            selected_indices=rand_idx,
            epochs=50,
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )

        results.append({
            "method": "Random",
            "budget": B,
            "seed": seed,
            "accuracy": acc,
            "runtime": runtime
        })

        np.save(f"unsupervised_results/loss_curves/random_B{B}_seed{seed}.npy",
                np.array(loss_curve))


In [ ]:
results = []

for seed in seeds:
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    for B in budgets:

        # --- TypiClust & TypiClust-HDBSCAN

        typi_idx = typiclust_kmeans[B]
        acc, loss_curve, runtime = train_supervised(
            selected_indices = typi_idx,
            epochs = 50, # 50 epochs of additional mounting following 500 epochs of training 
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )
        results.append({
            "method" : "TypiClust-KMeans",
            "budget" : B,
            "seed" : seed,
            "accuracy" : acc,
            "runtime" : runtime 
        })

        typi_idx = typiclust_hdbscan[B]
        acc, loss_curve, runtime = train_supervised(
            selected_indices=typi_idx,
            epochs=50,
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )
        results.append({
            "method" : "TypiClust-HDBSCAN",
            "budget" : B,
            "seed" : seed,
            "accuracy" : acc,
            "runtime" : runtime
        })


        #####################################

        # Save loss curve
        np.save(f"results/loss_curves/typiclust_B{B}_seed{seed}.npy", np.array(loss_curve))

        # ---   Random   ---

        rand_idx = random_selections[seed][B]
        acc, loss_curve, runtime = train_supervised(
            selected_indices=rand_idx,
            epochs=50,
            batch_size=128,
            lr=0.1,
            momentum=0.9,
            weight_decay=5e-4
        )

        results.append({
            "method": "Random",
            "budget": B,
            "seed": seed,
            "accuracy": acc,
            "runtime": runtime
        })

        np.save(f"results/loss_curves/random_B{B}_seed{seed}.npy", np.array(loss_curve))


# CSV Offloading 
df = pd.DataFrame(results)
df.to_csv("results/accuracies/all_results.csv", index=False)
df


df = pd.DataFrame(results)
df.to_csv("unsupervised_results/accuracies/all_results.csv", index=False)
df



In [ ]:
# Unsupervised vs other baselines
for method in df["method"].unique():
    sub = df[df["method"] == method]
    plt.plot(sub["budget"], sub["accuracy"], marker="o", label=method)

plt.xlabel("Budget")
plt.ylabel("Accuracy")
plt.title("Unsupervised TPCRP vs Random Baseline")
plt.legend()
plt.grid(True)
plt.savefig("unsupervised_results/accuracies/accuracy_curve.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.set(style="whitegrid", font_scale=1.2)

for method in ["TypiClust-KMeans", "TypiClust-HDBSCAN", "Random"]:
    means = []
    stds = []
    for B in budgets:
        subset = df[(df.method == method) & (df.budget == B)]
        means.append(subset.accuracy.mean())
        stds.append(subset.accuracy.std())

    plt.errorbar(budgets, means, yerr=stds, marker='o', capsize=4, label=method)


os.makedirs("results/plots", exist_ok=True)

plt.xlabel("Label Budget")
plt.ylabel("Test Accuracy")
plt.title("Accuracy vs Budget (CIFAR-10)")
plt.legend()
plt.savefig("results/plots/accuracy_vs_budget.png", dpi=300)
plt.show()


In [ ]:
plt.figure(figsize=(8,6))

for method in ["TypiClust-KMeans", "TypiClust-HDBSCAN"]:
    gain = []
    for B in budgets:
        t = df[(df.method==method) & (df.budget==B)].accuracy.mean()
        r = df[(df.method=="Random") & (df.budget==B)].accuracy.mean()
        gain.append(t-r)
    plt.plot(budgets, gain, marker="o", label=method)

plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Label Budget")
plt.ylabel("Accuracy Gain over Random")
plt.title("TypiClust Improvement over Random")
plt.legend()
plt.savefig("results/plots/gain_over_random.png", dpi=300)
plt.show()


In [ ]:
from scipy.stats import ttest_rel

# Either set method = method  
# Method = "TypiClust-KMeans" or "TypiClust-HDBSCAN

for method in ["TypiClust-KMeans", "TypiClust-HDBSCAN"]:
    print(f"\n=== {method} vs Random ===")
    for B in budgets:
        t = df[(df.method==method) & (df.budget==B)].accuracy.values
        r = df[(df.method=="Random") & (df.budget==B)].accuracy.values

        stat, p = ttest_rel(t, r)
        print(f"B={B}: p-value = {p:.4e}")



In [ ]:
ssl_loss = np.load("../TPCRP_Algorithm/modified_budget_results/ssl_loss.npy")


plt.figure(figsize=(8,6))
plt.plot(ssl_loss)
plt.xlabel("Epoch")
plt.ylabel("SSL Loss")
plt.title("Self-Supervised Training Loss (NT-Xent)")
plt.savefig("results/plots/ssl_loss_curve.png", dpi=300)
plt.show()


In [ ]:
from sklearn.manifold import TSNE

# Load features from Task 1
features = np.load("../TPCRP_Algorithm/modified_budget_results/features.npy")

tsne = TSNE(n_components=2, perplexity=30)
emb = tsne.fit_transform(features)

plt.figure(figsize=(8,8))
plt.scatter(emb[:,0], emb[:,1], s=2, alpha=0.5)
plt.title("t-SNE of SSL Features")
plt.savefig("results/plots/tsne.png", dpi=300)
plt.show()


In [ ]:
import matplotlib.pyplot as plt



def show_images(indices, title):
    dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True)
    plt.figure(figsize=(10,10))
    for i, idx in enumerate(indices[:25]):
        plt.subplot(5,5,i+1)
        plt.imshow(dataset[idx][0])
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

show_images(typiclust_hdbscan[20], "TypiClust-HDBSCAN Selected Samples (B=20)")
show_images(typiclust_kmeans[20], "TypiClust Selected Samples (B=20)")
show_images(random_selections[0][20], "Random Selected Samples (B=20)")


In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt  
import seaborn as sns  
import random 

import torch 

import torch.nn as nn 
from torch.utils.data import DataLoader, Subset


import torchvision 
import torchvision.transforms as transforms 

from sklearn.datasets import make_blobs
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans

from typing import List, Tuple

from sklearn.decomposition import PCA 


from Notebooks.supervised_training import train_supervised

In [ ]:
# Accuracy Calculations
def compute_accuracy_from_npy(npy_path, encoder, head_class, train_dataset, test_loader,
                              epochs=50, device="cuda"):

    # Load selected indices
    indices = np.load(npy_path).astype(int).tolist()

    # Build labeled subset
    subset = torch.utils.data.Subset(train_dataset, indices)
    loader = DataLoader(subset, batch_size=32, shuffle=True)

    # Create a fresh linear head
    head = head_class(in_dim=encoder.feat_dim, num_classes=10).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3)
    ce = nn.CrossEntropyLoss()

    encoder.eval()
    head.train()

    # Train supervised head
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                h = encoder.represent(x)
            logits = head(h)
            loss = ce(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()

    # Evaluate
    encoder.eval()
    head.eval()
    preds, trues = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            h = encoder.represent(x)
            logits = head(h)
            preds.append(logits.argmax(dim=1).cpu().numpy())
            trues.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    trues = np.concatenate(trues)

    acc = (preds == trues).mean() * 100
    return acc

In [ ]:
from TPCRP_Algorithm.semi_supervised_embeddings_TPCRP import LinearHead, ResNetEncd

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

encoder = ResNetEncd(base="resnet18", proj_dim=128).to(DEVICE)
encoder.load_state_dict(torch.load(
    "budget_results/semi_supervised_budget_Results/ssl_encoder.pth",
    map_location=DEVICE
))
encoder.eval()

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2470, 0.2435, 0.2616)),
])

cifar_train = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=base_transform
)
cifar_test = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=base_transform
)


test_loader = DataLoader(cifar_test, batch_size=256, shuffle=False)


acc_TPC_B10 = compute_accuracy_from_npy("budget_results/semi_supervised_budget_Results/typiclust_HDBSCAN_B10.npy",
                                    encoder, LinearHead, cifar_train, test_loader)

acc_balanced_B10 = compute_accuracy_from_npy("budget_results/semi_supervised_budget_Results/typiclust_HDBSCAN_B10.npy",
                                    encoder, LinearHead, cifar_train, test_loader)

In [ ]:
print(acc_TPC_B10) # For typiclust_HDBSCAN_ - that is TPCRP - Set for Budget = 10, for computational limitation related reasons